In [1]:
#Pandas groupby分组操作：拆分 ，应用，合并
#基本使用
#假设我们有一个销售数据集
import pandas as pd

data = {
    'Category': ['电子产品', '服装', '电子产品', '服装', '电子产品', '服装'],
    'Product': ['手机', '衬衫', '电脑', '裤子', '平板', '裙子'],
    'Sales': [5000, 1500, 8000, 1200, 3000, 900],
    'Region': ['华北', '华东', '华北', '华东', '华南', '华南']
}

df = pd.DataFrame(data)
print(df)

print('|||||||||')

#groupby本身若不进行聚合处理是无法输出的，若仅仅想分组，可以对其进行遍历以实现。
print(df.groupby('Region'))
for Category, Product in df.groupby('Region'):
    print(Category, Product)

  Category Product  Sales Region
0     电子产品      手机   5000     华北
1       服装      衬衫   1500     华东
2     电子产品      电脑   8000     华北
3       服装      裤子   1200     华东
4     电子产品      平板   3000     华南
5       服装      裙子    900     华南
|||||||||
华东   Category Product  Sales Region
1       服装      衬衫   1500     华东
3       服装      裤子   1200     华东
华北   Category Product  Sales Region
0     电子产品      手机   5000     华北
2     电子产品      电脑   8000     华北
华南   Category Product  Sales Region
4     电子产品      平板   3000     华南
5       服装      裙子    900     华南


In [2]:
#按单个列分组
#按照分组，在统计每组的平均销售额
gr = df.groupby('Category')['Sales'].mean()
print(gr)

Category
服装      1200.000000
电子产品    5333.333333
Name: Sales, dtype: float64


In [3]:
#按多个列分组
gr = df.groupby(['Category','Region'])['Sales'].sum()
print(gr)

Category  Region
服装        华东         2700
          华南          900
电子产品      华北        13000
          华南         3000
Name: Sales, dtype: int64


In [4]:
#对不同列使用不同的聚合函数
gr = df.groupby('Category').agg({'Sales':'sum'})   
#df.groupby('分组列').agg({'销售额': 'sum', '数量': 'mean'})，用字典指定每列算啥。这里是原始数据只有一列是数字才能算，所以只写了一个
print(gr)

          Sales
Category       
服装         3600
电子产品      16000


In [5]:
#自定聚合函数
def name_s(s):#计算销售额的范围
    return s.max() - s.min()

gr = df.groupby('Category')['Sales'].agg(name_s)
print(gr)

Category
服装       600
电子产品    5000
Name: Sales, dtype: int64


In [6]:
#分组后的迭代(遍历)
#groupby是可以进行迭代（遍历）
for name,gr in df.groupby('Category'):
    print(f"类别：{name}")
    print(gr)

类别：服装
  Category Product  Sales Region
1       服装      衬衫   1500     华东
3       服装      裤子   1200     华东
5       服装      裙子    900     华南
类别：电子产品
  Category Product  Sales Region
0     电子产品      手机   5000     华北
2     电子产品      电脑   8000     华北
4     电子产品      平板   3000     华南


In [7]:
# 分组后的过滤
#使用filter()方法根据条件过滤分组
#保留销售额总和大于5000的类别
dfs = df.groupby('Category').filter(lambda x: x['Sales'].sum() > 5000)
print(dfs)

  Category Product  Sales Region
0     电子产品      手机   5000     华北
2     电子产品      电脑   8000     华北
4     电子产品      平板   3000     华南


In [8]:
# 分组后的转换
#计算每个类别的销售额占比
df['kk'] = df.groupby('Category')['Sales'].transform(lambda x: x / x.sum())
print(df)

  Category Product  Sales Region        kk
0     电子产品      手机   5000     华北  0.312500
1       服装      衬衫   1500     华东  0.416667
2     电子产品      电脑   8000     华北  0.500000
3       服装      裤子   1200     华东  0.333333
4     电子产品      平板   3000     华南  0.187500
5       服装      裙子    900     华南  0.250000


In [9]:
#分组后的apply操作
#传入的是完整的DateFrame，而agg是对一列的聚合。所以前者明显是可以所更多的操作，比如这里就明显是对整体加了一列
def add_name(s):
    s['QQ'] = s['Sales'].rank(ascending = False)#rank分等级，ascending升序设置
    return s

dfs = df.groupby('Category').apply(add_name)
print(dfs)

           Product  Sales Region        kk   QQ
Category                                       
服装       1      衬衫   1500     华东  0.416667  1.0
         3      裤子   1200     华东  0.333333  2.0
         5      裙子    900     华南  0.250000  3.0
电子产品     0      手机   5000     华北  0.312500  2.0
         2      电脑   8000     华北  0.500000  1.0
         4      平板   3000     华南  0.187500  3.0


In [10]:
#Pandas merge合并操作
import pandas as pd 
#在单个键上进行合并操作
df = pd.DataFrame({'id':[1,2,3],'name':["张三","李四","王五"]})

df1 = pd.DataFrame({'id':[1,2,3],'name':["张二","李五","王刘"]})

df3 = pd.merge(df,df1,on='id')
df4 = pd.concat([df, df1], keys = ['src1', 'src2'])   #concat是直接叠上去，前者是合并。注：kyes可以实现标记
#通过on参数指定合并连接的键
print(df3)
print()
print(df4)

   id name_x name_y
0   1     张三     张二
1   2     李四     李五
2   3     王五     王刘

        id name
src1 0   1   张三
     1   2   李四
     2   3   王五
src2 0   1   张二
     1   2   李五
     2   3   王刘


In [11]:
#多个键上进行合并操作
import pandas as pd 
left = pd.DataFrame({ 
   'id':[1,2,3,4], 
   'Name': ['Smith', 'Maiki', 'Hunter', 'Hilen'], 
   'subject_id':['sub1','sub2','sub4','sub6']}) 

right = pd.DataFrame({ 
    'id':[1,2,3,4], 
   'Name': ['Bill', 'Lucy', 'Jack', 'Mike'], 
   'subject_id':['sub2','sub4','sub3','sub6']}) 
print(pd.merge(left,right,on=['id','subject_id']))

   id Name_x subject_id Name_y
0   4  Hilen       sub6   Mike


In [12]:
import pandas as pd 
left = pd.DataFrame({ 
   'id':[1,2,3,4], 
   'Name': ['Smith', 'Maiki', 'Hunter', 'Hilen'], 
   'subject_id':['sub1','sub2','sub4','sub6']}) 

right = pd.DataFrame({ 
    'id':[1,2,3,4], 
   'Name': ['Bill', 'Lucy', 'Jack', 'Mike'], 
   'subject_id':['sub2','sub4','sub3','sub6']}) 
print(pd.merge(left,right,on='subject_id',how = 'inner'))

   id_x  Name_x subject_id  id_y Name_y
0     2   Maiki       sub2     1   Bill
1     3  Hunter       sub4     2   Lucy
2     4   Hilen       sub6     4   Mike


In [13]:
#merge的合并是对指定键的内容的匹配以完成合并。how便是决定如何匹配和保留对应内容的

import numpy as np
import pandas as pd

left = pd.DataFrame({"sid":[1,2,3], "name":["A","B","C"]})
right = pd.DataFrame({"sid":[2,3,4], "score":[88,92,76]})

df_inner = pd.merge(left,right,on="sid",how="inner")    #inner：要合并的键以交集的形式保留，即只留一样的内容
df_left = pd.merge(left,right,on="sid",how="left")    #left：以左边的为主作为形式保留，即优先留住左边的内容，右边没有就补上NaN。right是同理的。
df_outer = pd.merge(left,right,on="sid",how="outer")    #outer：要合并的键以并集的形式保留，即全部留住，因此也称为全连接
print("内连接\n",df_inner)
print("左连接\n",df_left)
print("全外连接\n",df_outer)


内连接
    sid name  score
0    2    B     88
1    3    C     92
左连接
    sid name  score
0    1    A    NaN
1    2    B   88.0
2    3    C   92.0
全外连接
    sid name  score
0    1    A    NaN
1    2    B   88.0
2    3    C   92.0
3    4  NaN   76.0


In [23]:
#tansform相比于agg不会压缩内容，由不必像aplly那样自己写操作。适用于标准化数据（如下）
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "group": ["A", "A", "A", "B", "B", "B"],
    "value": [1, 2, 3, 10, 20, 30],
})

# 按 group 分组后做标准化，transform 会保持原索引和长度
result = df.groupby("group")["value"].transform(
    lambda s: (s - s.mean()) / s.std()
)

expected = pd.Series(
    [-1.0, 0.0, 1.0, -1.0, 0.0, 1.0],
    name="value",
)

print(result)
print(expected)

0   -1.0
1    0.0
2    1.0
3   -1.0
4    0.0
5    1.0
Name: value, dtype: float64
0   -1.0
1    0.0
2    1.0
3   -1.0
4    0.0
5    1.0
Name: value, dtype: float64


In [24]:
df = pd.DataFrame({
    "a": [1, 2, 3],
    "b": [4, 5, 6],
})

result = df.transform(lambda x: x * 2)

expected = pd.DataFrame({
    "a": [2, 4, 6],
    "b": [8, 10, 12],
})

print(result)
print(expected)
print(result.compare(expected))    #返回差异值，df1.equal(df2)只能返回对应的一个布尔值

   a   b
0  2   8
1  4  10
2  6  12
   a   b
0  2   8
1  4  10
2  6  12
Empty DataFrame
Columns: []
Index: []
